# Waveform plotting

Input is a directory of NPZ parts, or the single NPZ from notebook 01.
No manifest is read or required. Source IDs and sample offsets are read from
the parts, sorted, and checked for gaps, overlaps, and the full billion samples.
File hashes, segment accounting, and inset completeness are checked; full-record
integer moments are saved for independent verification by notebook 03. Memory stays
bounded; part boundaries preserve every connecting segment.

Dependencies: Python 3.10+, NumPy, Numba, SciPy, Matplotlib and PyMuPDF.
No repository Python modules are imported. Run from the repository directory
or edit the paths below. Existing derived plots are replaced on rerun.

The center of each recording is measured as its exact integer sum divided
by its sample count and computed during waveform rendering. No nominal source
centers are supplied in the settings.


In [ ]:
from __future__ import annotations
import hashlib
import io
import json
import math
import os
from pathlib import Path
import tempfile
import time
import zipfile
from dataclasses import dataclass
import numpy as np

import csv
from numba import njit
from scipy.special import ndtr, log_ndtr
VERSION = '13.0'
MIN_FONT_SIZE = 18.0
DEFAULT_BRIGHTNESS_GAMMA = DEFAULT_BRIGHTNESS_EXPOSURE = 1.0
BRIGHTNESS_REFERENCE_PERCENTILE = 99.9
VISIBLE_SOURCES = (0, 1, 2, 3)
SUPPORT_MODES = ('sample_count', 'independent_windows')


In [ ]:
INPUT = Path('waveform_parts')  # Or fresh_run3/complete_waveforms.npz
OUTPUT_DIR = Path('./')
ANALYSIS_CHUNK = 4_000_000
VERIFY_CHECKSUMS = True

# All settings needed for the golden figure; no generation metadata is read.
SETTINGS = {'sample_rate_hz': 10000.0,
 'points_per_recording': [1000000000, 1000000000, 1000000000, 1000000000],
 'main_view': {'start': 0,
               'stop': 1000000000,
               'nx': 1600,
               'ny': 1000,
               'ymin': -35.0,
               'ymax': 5.0},
 'inset_view': {'start': 540000000, 'stop': 540000400, 'ymin': -15.0, 'ymax': 5.0},
 'main_y_ticks_codes': [-35, -30, -25, -20, -15, -10, -5, 0, 5],
 'main_panel_sources': [0, 1, 2, 3],
 'delta_c_ff': [0.0, 50.0, 75.0, 100.0],
 'waveform_threshold_codes': -6.35,
 'render_settings': {'gamma': 0.8, 'exposure': 1.0},
 'analysis': {'support_mode': 'sample_count'}}


Dataset reader and exact connected-line rasterizer.


In [ ]:

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024**2), b''):
            digest.update(block)
    return digest.hexdigest()


class WaveformInput:
    """Discover self-describing NPZ parts; no manifest or generation report is needed."""
    def __init__(self, path, verify_checksums=True, settings=None):
        self.path = Path(path).resolve()
        self.verify_checksums = verify_checksums
        self.metadata = json.loads(json.dumps(SETTINGS if settings is None else settings))
        self.parts = None
        counts = self.metadata['points_per_recording']
        if len(counts) != 4:
            raise ValueError('Four recordings are required.')
        if self.path.is_dir():
            self.parts = []
            for path in sorted(self.path.glob('*.npz')):
                with np.load(path, allow_pickle=False) as archive:
                    sid_array, start_array = archive['source_index'], archive['start_sample']
                    if (sid_array.shape != () or start_array.shape != ()
                            or sid_array.dtype.kind not in 'iu' or start_array.dtype.kind not in 'iu'):
                        raise ValueError(f'Source ID and sample offset must be integer scalars: {path}')
                    sid, start = int(sid_array.item()), int(start_array.item())
                with zipfile.ZipFile(path) as archive:
                    with archive.open('waveform.npy') as handle:
                        shape, order, dtype = self.array_header(handle)
                if (sid not in range(4) or start < 0 or len(shape) != 1 or shape[0] < 1
                        or order or dtype != np.dtype('<i2')):
                    raise ValueError(f'Invalid waveform part: {path}')
                self.parts.append(dict(file=path.name, source_index=sid, start=start,
                    count=shape[0], bytes=path.stat().st_size, sha256=sha256_file(path)))
            self.parts.sort(key=lambda p: (p['source_index'], p['start']))
            for sid, count in enumerate(counts):
                expected = 0
                for part in (p for p in self.parts if p['source_index'] == sid):
                    if part['start'] != expected:
                        raise ValueError(f'Source {sid}: missing, duplicate or overlapping part at sample {expected:,}.')
                    expected += part['count']
                if expected != count:
                    raise ValueError(f'Source {sid}: found {expected:,} samples; expected {count:,}.')
            # Hash source order, ranges and actual file contents, independently
            # of filenames. Notebook 03 compares this with notebook 02's output.
            identity = [{key: part[key] for key in ('source_index','start','count','sha256')}
                        for part in self.parts]
            self.fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
        elif self.path.is_file() and self.path.suffix.lower() == '.npz':
            with zipfile.ZipFile(self.path) as archive:
                for sid, count in enumerate(counts):
                    with archive.open(f'source_{sid}.npy') as handle:
                        shape, order, dtype = self.array_header(handle)
                    if shape != (count,) or order or dtype != np.dtype('<i2'):
                        raise ValueError(f'Invalid source {sid} array header.')
            # Any legacy metadata member is deliberately ignored.
            self.fingerprint = sha256_file(self.path)
        else:
            raise ValueError('INPUT must be a directory of NPZ parts or a single waveform NPZ file.')

    @staticmethod
    def array_header(handle):
        version = np.lib.format.read_magic(handle)
        if version == (1, 0):
            return np.lib.format.read_array_header_1_0(handle)
        if version == (2, 0):
            return np.lib.format.read_array_header_2_0(handle)
        raise ValueError(f'Unsupported NPY header version: {version}')

    @staticmethod
    def read_exact(handle, count):
        data = bytearray(count)
        view = memoryview(data)
        position = 0
        while position < count:
            amount = handle.readinto(view[position:])
            if not amount:
                raise EOFError('Truncated waveform array.')
            position += amount
        return data

    def pieces(self, sid, chunk):
        if self.parts is not None:
            for part in (p for p in self.parts if p['source_index'] == sid):
                path = self.path / part['file']
                if self.verify_checksums and sha256_file(path) != part['sha256']:
                    raise ValueError(f'Waveform part changed while processing: {path}')
                with np.load(path, allow_pickle=False) as archive:
                    values = archive['waveform']
                    if (values.shape != (part['count'],) or values.dtype != np.dtype('<i2')
                            or archive['source_index'].item() != sid
                            or archive['start_sample'].item() != part['start']):
                        raise ValueError(f'Incorrect waveform header: {path}')
                yield values
        else:
            with zipfile.ZipFile(self.path) as archive:
                with archive.open(f'source_{sid}.npy') as handle:
                    shape, order, dtype = self.array_header(handle)
                    count = self.metadata['points_per_recording'][sid]
                    if shape != (count,) or order or dtype != np.dtype('<i2'):
                        raise ValueError(f'Incorrect source {sid} array header.')
                    for start in range(0, count, chunk):
                        size = min(chunk, count-start)
                        yield np.frombuffer(self.read_exact(handle, 2*size), dtype='<i2')
                    if handle.read(1):
                        raise ValueError('Unexpected trailing waveform bytes.')

    def blocks(self, sid, chunk):
        """Reblock any part boundaries to the original analysis chunk size."""
        if chunk < 1:
            raise ValueError('Chunk size must be positive.')
        buffer = np.empty(chunk, dtype='<i2')
        filled = 0
        for piece in self.pieces(sid, chunk):
            offset = 0
            while offset < piece.size:
                amount = min(chunk-filled, piece.size-offset)
                buffer[filled:filled+amount] = piece[offset:offset+amount]
                filled += amount
                offset += amount
                if filled == chunk:
                    yield buffer
                    buffer = np.empty(chunk, dtype='<i2')
                    filled = 0
        if filled:
            yield buffer[:filled]


In [ ]:
@dataclass(frozen=True)
class View:
    """Time interval [start,stop); amplitude limits are inclusive."""
    start: int
    stop: int
    nx: int
    ny: int
    ymin: float
    ymax: float

    def validate(self) -> None:
        if not (0 <= self.start < self.stop <= np.iinfo(np.int64).max):
            raise ValueError("A view needs 0 <= start < stop within int64 range.")
        if not (1 <= self.nx <= 65536 and 1 <= self.ny <= 65536):
            raise ValueError("Each histogram dimension must be in [1,65536].")
        if (self.stop - self.start) * self.nx > np.iinfo(np.int64).max:
            raise ValueError("Time span times horizontal bins exceeds int64.")
        _validate_amplitude(self.ymin, self.ymax)
        if not math.isfinite(self.ny / (self.ymax - self.ymin)):
            raise ValueError("Amplitude scale is not representable.")

    @property
    def cells(self) -> int:
        return self.nx * self.ny

    def kernel_args(self) -> tuple:
        return (np.int64(self.start), np.int64(self.stop), np.int32(self.nx),
                np.int32(self.ny), np.float64(self.ymin), np.float64(self.ymax))


@dataclass(frozen=True)
class Inset:
    start: int
    stop: int
    ymin: float
    ymax: float

    @property
    def count(self) -> int:
        return self.stop - self.start

    def validate(self) -> None:
        if not (0 <= self.start < self.stop <= np.iinfo(np.int64).max):
            raise ValueError("Inset sample interval must be nonempty and within int64.")
        if self.count > 1_000_000:
            raise ValueError("Line inset is limited to 1,000,000 samples; default is 200.")
        _validate_amplitude(self.ymin, self.ymax)


def _validate_amplitude(lo: float, hi: float) -> None:
    if not (math.isfinite(lo) and math.isfinite(hi) and lo < hi
            and math.isfinite(hi - lo)):
        raise ValueError("Amplitude limits must be finite and increasing.")
def _cpu_lines_impl(samples, core_count, start, sid, gain, bias, quantize,
                    m0, m1, mx, my, mlo, mhi, delta, z0, z1, traces):
    """Exact Bresenham pixel counts via column difference arrays.

    A vertical run ylo..yhi is delta[x,ylo] += 1, delta[x,yhi+1] -= 1.
    Prefix sums later fill ALL its pixels. This is exactly equivalent to
    incrementing each pixel, but avoids billions of long vertical loops.
    Both endpoints count once per segment, including zero-length segments.
    """
    scale = my / (mhi - mlo)
    span, pitch = m1 - m0, my + 1
    drawn = bad = outside = vertical = general = 0
    for k in range(core_count):
        index = start + k
        a = np.float64(samples[k]) * gain + bias
        if quantize:
            a = np.rint(a)
        if z0 <= index < z1:
            traces[sid, index - z0] = a
        if k + 1 >= samples.size or index < m0 or index + 1 >= m1:
            continue
        b = np.float64(samples[k+1]) * gain + bias
        if quantize:
            b = np.rint(b)
        if not (np.isfinite(a) and np.isfinite(b)):
            bad += 1
            continue
        if (a < mlo and b < mlo) or (a > mhi and b > mhi):
            outside += 1
            continue
        q = (index - m0) * mx
        x0, x1 = q // span, (q + mx) // span
        # Clip in data space BEFORE discretizing. In particular, reject an
        # entirely out-of-range segment instead of smearing it along a border.
        if x0 != x1 and not (mlo <= a <= mhi and mlo <= b <= mhi):
            t0, t1 = 0.0, 1.0
            dy = b - a
            if np.isfinite(dy):
                if a < mlo:
                    t0 = (mlo-a)/dy
                elif a > mhi:
                    t0 = (mhi-a)/dy
                if b < mlo:
                    t1 = (mlo-a)/dy
                elif b > mhi:
                    t1 = (mhi-a)/dy
            else:
                # Finite opposite-sign values can have an overflowing b-a.
                magnitude = max(abs(a), abs(b))
                aa, bb = a/magnitude, b/magnitude
                if a < mlo:
                    t0 = (mlo/magnitude-aa)/(bb-aa)
                elif a > mhi:
                    t0 = (mhi/magnitude-aa)/(bb-aa)
                if b < mlo:
                    t1 = (mlo/magnitude-aa)/(bb-aa)
                elif b > mhi:
                    t1 = (mhi/magnitude-aa)/(bb-aa)
            base, remainder = x0, q % span
            if t0 > 0:
                x0 = min(mx-1, base + int((remainder + t0*mx)/span))
            if t1 < 1:
                x1 = min(mx-1, base + int((remainder + t1*mx)/span))
        a, b = max(mlo, min(a, mhi)), max(mlo, min(b, mhi))
        y0 = min(my-1, int((a-mlo)*scale))
        y1 = min(my-1, int((b-mlo)*scale))
        drawn += 1
        if x0 == x1:
            lo, hi = min(y0, y1), max(y0, y1)
            delta[x0*pitch+lo] += np.int64(1)
            delta[x0*pitch+hi+1] -= np.int64(1)
            vertical += 1
        else:
            general += 1
            dx, dy = x1-x0, -abs(y1-y0)
            sy = 1 if y0 < y1 else -1
            error = dx + dy
            # Compress the Bresenham pixels into one vertical run per column.
            # The exact traversal/tie-breaking is unchanged.
            x, y, run_start = x0, y0, y0
            while True:
                if x == x1 and y == y1:
                    lo, hi = min(run_start, y), max(run_start, y)
                    delta[x*pitch+lo] += np.int64(1)
                    delta[x*pitch+hi+1] -= np.int64(1)
                    break
                twice_error = 2*error
                step_x = twice_error >= dy
                if step_x:
                    lo, hi = min(run_start, y), max(run_start, y)
                    delta[x*pitch+lo] += np.int64(1)
                    delta[x*pitch+hi+1] -= np.int64(1)
                    error += dy
                    x += 1
                if twice_error <= dx:
                    error += dx
                    y += sy
                if step_x:
                    run_start = y
    return drawn, bad, outside, vertical, general


def _finish_difference(delta: np.ndarray, main: View) -> np.ndarray:
    prefix = np.cumsum(delta.reshape(main.nx, main.ny+1), axis=1, dtype=np.int64)
    if np.any(prefix[:, -1] != 0) or np.any(prefix[:, :-1] < 0):
        raise RuntimeError("Invalid line difference counts (or signed counter overflow).")
    return prefix[:, :-1].T.astype(np.uint64)

line_kernel = njit(cache=False, nogil=True)(_cpu_lines_impl)


In [ ]:
def raster_waveforms(reader, chunk=4_000_000):
    metadata = reader.metadata
    main, inset = View(**metadata['main_view']), Inset(**metadata['inset_view'])
    main.validate()
    inset.validate()
    if any(n != main.stop for n in metadata['points_per_recording']) or main.start != 0:
        raise ValueError('The golden waveform panel requires full equal-length recordings.')
    hist = np.empty((4, main.ny, main.nx), dtype=np.uint64)
    traces = np.empty((4, inset.count))
    line_counts = []
    metadata["source_checks"] = []
    metadata["centers_codes"] = []
    for sid, count in enumerate(metadata['points_per_recording']):
        if inset.stop > count:
            raise ValueError('Inset is outside the recording.')
        delta = np.zeros(main.nx*(main.ny+1), dtype=np.int64)
        trace = np.full((1, inset.count), np.nan)
        lines = np.zeros(5, dtype=np.uint64)
        total = squared = start = 0
        blocks = iter(reader.blocks(sid, chunk))
        current = next(blocks)
        while current is not None:
            following = next(blocks, None)
            values = np.concatenate((current, following[:1])) if following is not None else current
            integers = current.astype(np.int64)
            total += int(integers.sum(dtype=np.int64))
            squared += int(np.square(integers).sum(dtype=np.int64))
            stats = line_kernel(values, np.int64(current.size), np.int64(start), np.int64(0),
                np.float64(1.), np.float64(0.), np.int32(1), *main.kernel_args(), delta,
                np.int64(inset.start), np.int64(inset.stop), trace)
            lines += np.asarray(stats, dtype=np.uint64)
            start += current.size
            current = following
            if (start // chunk) % 25 == 0:
                print(f'Source {sid}: rasterized {start:,}/{count:,}', flush=True)
        assert start == count
        metadata['centers_codes'].append(total / count)
        metadata['source_checks'].append(dict(samples=count, integer_sum=total, integer_sum_of_squares=squared))
        assert sum(int(v) for v in lines[:3]) == count-1
        assert lines[1] == 0 and np.isfinite(trace).all()
        hist[sid], traces[sid] = _finish_difference(delta, main), trace[0]
        line_counts.append(lines.tolist())
    return hist, traces, line_counts


Golden waveform styling, with the same physical panel and inset dimensions.


In [ ]:
def _short_number(value: float) -> str:
    for scale, suffix in ((1e9, "G"), (1e6, "M"), (1e3, "k")):
        if abs(value) >= scale:
            return f"{value/scale:g}{suffix}"
    return f"{value:g}"


def phosphor_opacity(hist: np.ndarray,
                     gamma: float = DEFAULT_BRIGHTNESS_GAMMA,
                     exposure: float = DEFAULT_BRIGHTNESS_EXPOSURE) -> np.ndarray:
    """Map line-hit density to color opacity; the default is linear (gamma=1).

    Let ref = max(1, percentile(positive_counts, 99.9)) and
    x = clip(counts / ref, 0, 1). Return clip(exposure * x**gamma, 0, 1).
    With gamma=1 and exposure=1, 10% density gives 10% opacity and
    50% density gives 50% opacity. Empty pixels remain white after compositing.

    This replaces the old 1-exp(-exposure*x**gamma) transfer entirely.
    The normalization percentile is unchanged; counts above ref saturate.
    Computation uses only the small density array and does not modify it.
    No graphics backend is imported and no figure or image is created here.
    """
    if not (math.isfinite(gamma) and gamma > 0
            and math.isfinite(exposure) and exposure > 0):
        raise ValueError("Gamma and exposure must be finite and positive.")
    values = np.asarray(hist)
    if values.dtype.kind not in "uif":
        raise ValueError("Histogram counts must be real numeric values.")
    if np.any(values < 0) or not np.all(np.isfinite(values)):
        raise ValueError("Histogram counts must be finite and nonnegative.")
    positive = values[values > 0]
    ref = (max(1.0, float(np.percentile(positive, BRIGHTNESS_REFERENCE_PERCENTILE)))
           if positive.size else 1.0)
    # float64 avoids needless precision loss in the uint64 normalization.
    # Work on a copy so saved density counts remain reusable and unchanged.
    alpha = values.astype(np.float64, copy=True)
    alpha /= ref
    np.clip(alpha, 0.0, 1.0, out=alpha)
    np.power(alpha, gamma, out=alpha)
    alpha *= exposure
    np.clip(alpha, 0.0, 1.0, out=alpha)
    return alpha


def phosphor_rgb(hist: np.ndarray, color: str,
                 gamma: float = DEFAULT_BRIGHTNESS_GAMMA,
                 exposure: float = DEFAULT_BRIGHTNESS_EXPOSURE) -> np.ndarray:
    """Single-color power-law opacity on white; ONE opaque PDF image XObject."""
    from matplotlib.colors import to_rgb
    alpha = phosphor_opacity(hist, gamma=gamma, exposure=exposure)
    rgb = 1.0 - alpha[..., None] * (1.0 - np.asarray(to_rgb(color), dtype=np.float64))
    return np.rint(np.clip(rgb, 0, 1) * 255).astype(np.uint8)


def font_settings(font_files: list[str] | None = None) -> tuple[dict, dict]:
    """Use PDF core Helvetica, or embed user-owned Helvetica TTF/OTF files."""
    settings = {"font.family": "sans-serif", "font.sans-serif": ["Helvetica"],
                "pdf.use14corefonts": True, "pdf.fonttype": 42, "ps.fonttype": 42,
                "text.usetex": False, "text.parse_math": False,
                "axes.unicode_minus": False, "path.simplify": False,
                "font.size": MIN_FONT_SIZE, "axes.labelsize": MIN_FONT_SIZE,
                "axes.titlesize": MIN_FONT_SIZE, "figure.titlesize": MIN_FONT_SIZE,
                "xtick.labelsize": MIN_FONT_SIZE, "ytick.labelsize": MIN_FONT_SIZE,
                "legend.fontsize": MIN_FONT_SIZE, "legend.title_fontsize": MIN_FONT_SIZE}
    info = {"family": "Helvetica", "mode": "PDF Base-14 Helvetica (not embedded)",
            "text_as_paths": False, "bitmap": "overview only", "inset": "vector zero-order hold",
            "minimum_font_size_pt": MIN_FONT_SIZE, "main_tick_font_size_pt": MIN_FONT_SIZE,
            "inset_tick_font_size_pt": MIN_FONT_SIZE, "inset_xlabel_font_size_pt": MIN_FONT_SIZE}
    if font_files:
        from matplotlib import font_manager
        families = []
        for filename in font_files:
            prop = font_manager.FontProperties(fname=filename)
            family = prop.get_name()
            if "helvetica" not in family.lower():
                raise ValueError(f"{filename}: font is {family!r}, not Helvetica.")
            font_manager.fontManager.addfont(filename)
            families.append(family)
        settings.update({"font.sans-serif": list(dict.fromkeys(families)), "pdf.use14corefonts": False})
        info["mode"] = "User-supplied Helvetica, subset-embedded (pdf.fonttype=42)"
    return settings, info


def _hour_axis_ticks(start: float, stop: float) -> tuple[np.ndarray, list[str]]:
    """Readable hour ticks including the true endpoints, without a crowded last tick."""
    from matplotlib.ticker import MaxNLocator
    if not (math.isfinite(start) and math.isfinite(stop) and start < stop):
        raise ValueError("Hour-axis bounds must be finite and increasing.")
    locator = MaxNLocator(nbins=6, min_n_ticks=3, steps=[1, 2, 2.5, 5, 10])
    candidates = locator.tick_values(start, stop)
    interior = candidates[(candidates > start) & (candidates < stop)]
    spacing = float(np.median(np.diff(candidates)))
    # Keep endpoint labels, omitting a nearby interior tick instead of overlapping.
    if interior.size and interior[0] - start < 0.55 * spacing:
        interior = interior[1:]
    if interior.size and stop - interior[-1] < 0.55 * spacing:
        interior = interior[:-1]
    ticks = np.concatenate(([start], interior, [stop]))
    minimum_gap = float(np.min(np.diff(ticks)))
    precision = max(2, 1 - math.floor(math.log10(minimum_gap)))
    labels = [f"{value:.{precision}f}".rstrip("0").rstrip(".") for value in ticks]
    return ticks, labels


def preview_pdf(pdf: str | Path, png: str | Path, dpi: int = 160) -> None:
    """Optional preview uses the actual PDF so Helvetica core-font metrics agree."""
    try:
        import fitz
    except ImportError as exc:
        raise RuntimeError("--preview requires PyMuPDF: python -m pip install pymupdf") from exc
    path = Path(png)
    if path.suffix.lower() != ".png":
        raise ValueError("Preview filename must end in .png.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with fitz.open(pdf) as doc:
        doc[0].get_pixmap(dpi=dpi).save(path)
def render_waveform(hist, traces, metadata, output, *, color='#2400ff', gamma=1., exposure=1., font_files=None):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import to_rgb
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle, ConnectionPatch
    from matplotlib.text import Text
    from matplotlib.ticker import MaxNLocator, NullLocator
    output = Path(output)
    if output.suffix.lower() != '.pdf':
        raise ValueError('The study output must be PDF.')
    main, inset_view = View(**metadata['main_view']), Inset(**metadata['inset_view'])
    fs = metadata['sample_rate_hz']
    # Recompute model probabilities from the current paired-window statistics
    # on every render, including archive replots. Stored curve values are never
    # used as a historical reference or retained across a change in d-prime.
    support_mode = metadata.get('analysis', {}).get('support_mode', 'sample_count')
    support_label = '1/N' if support_mode == 'sample_count' else '1/window count'
    waveform_threshold = metadata.get('waveform_threshold_codes', 5.)
    curve_segments = {}
    # One source/color table drives the density, inset, and analysis panels.
    # A legacy 2-D archive contains merged counts and cannot be recolored by
    # source; preserve its original single-color rendering when replotting it.
    source_colors = ('#000000', '#0072BD', '#D95319', '#009E73')
    density = np.asarray(hist)
    separate_density = density.ndim == 3
    visible_sources = tuple(metadata.get('main_panel_sources',
                            range(4) if separate_density else VISIBLE_SOURCES))
    if not visible_sources or any(sid not in range(4) for sid in visible_sources):
        raise ValueError('Visible source indices must be drawn from the four study recordings.')
    if separate_density:
        if density.shape != (4, main.ny, main.nx):
            raise ValueError('Per-source density must have shape (4, main.ny, main.nx).')
        rgb = np.ones((main.ny, main.nx, 3), dtype=np.float64)
        for sid in visible_sources:
            opacity = phosphor_opacity(density[sid], gamma=gamma, exposure=exposure)[..., None]
            rgb = rgb * (1.0 - opacity) + np.asarray(to_rgb(source_colors[sid])) * opacity
        waveform_rgb = np.rint(np.clip(rgb, 0.0, 1.0) * 255).astype(np.uint8)
    else:
        if density.shape != (main.ny, main.nx):
            raise ValueError('Legacy density must have shape (main.ny, main.nx).')
        waveform_rgb = phosphor_rgb(density, color, gamma=gamma, exposure=exposure)

    rc, font_info = font_settings(font_files)
    with plt.rc_context(rc):
        fig = plt.figure(figsize=(10.6, 7.0), facecolor='white')
        try:
            ax = fig.add_axes([.061*22.8/10.6, .185, .350*22.8/10.6, .735])
            extent = (main.start/fs/3600, main.stop/fs/3600, main.ymin, main.ymax)
            ax.imshow(waveform_rgb,
                      extent=extent, origin='lower', aspect='auto', interpolation='none')
            ax.set_xlim(extent[:2]); ax.set_ylim(extent[2:])
            ticks, labels = _hour_axis_ticks(*extent[:2])
            ax.set_xticks(ticks, labels)
            ax.set_yticks(metadata.get('main_y_ticks_codes', [-30, -20, -10, 0, 10, 20]))
            ax.set_xlabel('Time / hours', fontsize=26, fontweight='bold', fontstyle='italic', labelpad=13)
            ax.set_ylabel(f'Output code at {_short_number(fs)}S/s', fontsize=25, fontweight='bold', fontstyle='italic', labelpad=13)
            # Threshold is a vector guide, not a third waveform.
            ax.axhline(waveform_threshold, color='black', linestyle='--', linewidth=1.2, alpha=.65)
            pos = ax.get_position()
            ins = fig.add_axes([pos.x0+.285*pos.width, pos.y0+.13*pos.height,
                                .665*pos.width, .27*pos.height], zorder=5, facecolor='white')
            plt.sca(ins)
            for sid in visible_sources:
                # Sample n is held on [n, n+1); the last sample is held through
                # the exclusive right boundary instead of ending at count-1.
                trace = np.asarray(traces[sid])
                if trace.shape != (inset_view.count,):
                    raise ValueError('Inset trace length does not match its sample window.')
                ins.step(np.arange(inset_view.count + 1), np.r_[trace, trace[-1]],
                         where='post', color=source_colors[sid] if separate_density else color, linewidth=.8,
                         solid_capstyle='butt', rasterized=False)
            ins.set_xlim(0, inset_view.count)
            ins.set_ylim(inset_view.ymin, inset_view.ymax)
            offsets = [0, 50, 100, 150, 200] if inset_view.count==200 else np.unique(np.rint(np.linspace(0, inset_view.count, 5)).astype(int))
            ins.set_xticks(offsets, [f'{inset_view.start/fs/3600:g} h']+[f'+{i}' for i in offsets[1:]])
            ins.set_xlabel('Sample Offset', fontsize=18, labelpad=5)
            ins.yaxis.set_major_locator(MaxNLocator(nbins=3, integer=True))
            z0, z1 = inset_view.start/fs/3600, inset_view.stop/fs/3600
            ax.add_patch(Rectangle((z0, inset_view.ymin), z1-z0, inset_view.ymax-inset_view.ymin,
                                   fill=False, edgecolor='black', linewidth=1.7, zorder=4))
            for x, corner in ((z0, (0, 1)), (z1, (1, 1))):
                ax.add_artist(ConnectionPatch(xyA=(x, inset_view.ymin), coordsA='data', axesA=ax,
                               xyB=corner, coordsB='axes fraction', axesB=ins, color='black', linewidth=1.7, zorder=4))
            for text in ins.get_yticklabels():
                text.set_bbox(dict(facecolor='white', edgecolor='none', pad=.5))
            for axis in (ax, ins):
                axis.tick_params(labelsize=18, length=6, width=1.3, pad=5)
                for spine in axis.spines.values():
                    spine.set_linewidth(1.35)
            for text in fig.findobj(match=Text):
                if text.get_fontsize()<18:
                    text.set_fontsize(18)
            output.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(output, format='pdf', dpi=200, facecolor='white',
                        metadata=dict(Title='Connected-line DPO and midpoint-threshold capacitance detection',
                                      Creator=f'phosphor.py {VERSION} / capacitance_study.py / Matplotlib',
                                      Subject='Bitmap waveform overview; vector axes, zero-order-hold inset, separability and Gaussian false-positive curves'))
        finally:
            plt.close(fig)
    return font_info


In [ ]:
reader = WaveformInput(INPUT, verify_checksums=VERIFY_CHECKSUMS)
hist, traces, line_counts = raster_waveforms(reader, ANALYSIS_CHUNK)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metadata = json.loads(json.dumps(reader.metadata))
metadata['waveform_input_sha256'] = reader.fingerprint
metadata['execution'] = dict(sample_decimation=False, samples_rasterized=sum(metadata['points_per_recording']),
    raster_chunk=ANALYSIS_CHUNK, line_statistics_per_source=line_counts)
np.savez_compressed(OUTPUT_DIR / 'study_density.npz', hist=hist, traces=traces)
metadata['density_sha256'] = sha256_file(OUTPUT_DIR / 'study_density.npz')
for sid in range(4):
    np.savez_compressed(OUTPUT_DIR / f'source_{sid}_density.npz', hist=hist[sid], traces=traces[sid])
    (OUTPUT_DIR / f'source_{sid}_raster.json').write_text(json.dumps(dict(
        source_index=sid, connected_line_statistics=line_counts[sid]), indent=2)+'\n')
pdf = OUTPUT_DIR / 'waveform_panel.pdf'
render_waveform(hist, traces, metadata, pdf, **metadata['render_settings'])
preview_pdf(pdf, pdf.with_suffix('.png'), dpi=130)
print(f'Saved waveform density and panel to {OUTPUT_DIR.resolve()}')

try:
    from IPython.display import Image, display
except ImportError:
    print(f'Preview: {pdf.with_suffix(".png").resolve()}')
else:
    display(Image(filename=str(pdf.with_suffix('.png'))))
